# TDA: HAN + GTN + PDGNN 파이프라인 (Colab)

이종 그래프(ACM)에서 **위상 특징(EPD)** 을 자동으로 뽑아 노드 분류에 쓰는 파이프라인을
Colab 에서 끝까지 돌려봅니다.

```
GTN(메타패스 자동 발견) → PDGNN(채널별 EPD) → semantic attention fusion → HAN 분류
```

- 코드 본체: https://github.com/jjune5/TDA
- GPU 런타임 권장 (런타임 → 런타임 유형 변경 → GPU).

> 참고: 성능 주장이 아니라 동작/실측 확인용입니다. 자세한 충실도·가정은 본 저장소 README 참고.

## 1. 설치

Colab 의 torch 에 맞춰 PyG 와 gudhi 를 설치합니다. (원본 PDGNN 의 `torch_scatter`,
persistence image 의 cython 의존성은 제거돼 있어 추가 빌드가 필요 없습니다.)

In [ ]:
!pip install -q torch_geometric gudhi
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
!git clone -q https://github.com/jjune5/TDA.git
%cd TDA
!pip install -q -e .

## 2. 데이터

ACM(HGB benchmark)은 `--data-root` 에 없으면 PyG 가 자동 다운로드합니다.
타겟 = paper(3 클래스), N=3025, 특징 1902 차원, train 770 / val 137 / test 2118.

In [ ]:
from tda.data import get_dataset
from tda.utils import load_json
cfg = load_json('configs/acm.json')
b = get_dataset('acm', cfg, data_root='./data')
print('N', b.num_nodes, 'feat', tuple(b.x.shape), 'classes', b.num_classes)
print('masks', {k: int(v.sum()) for k, v in b.masks.items()})
print('base_relations', list(b.base_relations))
print('han_metapaths', list(b.han_metapaths))

## 3. 전체 파이프라인 실행 (GTN + PDGNN + HAN, attention fusion)

GPU 에서 수 분 정도 걸립니다 (Stage 2 의 노드별 EPD 추출이 가장 오래 걸림).

In [ ]:
from tda.train import run
rec_full = run(cfg, 'acm', data_root='./data', output_dir='runs/acm_full')
print('\n[full] test Macro-F1 =', round(rec_full['test_macro_f1'], 4),
      '| acc =', round(rec_full['test_accuracy'], 4))

## 4. Baseline (HAN 단독, 위상 없음) 과 비교

In [ ]:
cfg_base = load_json('configs/acm.json'); cfg_base['use_topology'] = False
rec_base = run(cfg_base, 'acm', data_root='./data', output_dir='runs/acm_base')
print('\n=== ACM test Macro-F1 ===')
print('baseline (HAN)           :', round(rec_base['test_macro_f1'], 4))
print('full (GTN+PDGNN+HAN, attn):', round(rec_full['test_macro_f1'], 4))

## 5. GTN 이 발견한 메타패스 / fusion 가중치 살펴보기

In [ ]:
print('기저 관계 순서:', list(b.base_relations) + ['identity'])
print('GTN 레이어별 채널 어텐션 (관계 가중치):')
for li, layer in enumerate(rec_full.get('gtn_attentions') or []):
    print(f'  layer {li}:', [[round(x, 3) for x in ch] for ch in layer])
print('fusion β (채널 위상 특징 가중치):', [round(x, 3) for x in rec_full.get('fusion_beta', [])])

## 메모

- 위 수치는 단일 시드 실측입니다. 견고성(분산) 분석은 여러 시드로 반복하세요
  (`cfg['seed']` 변경).
- 데이터셋 교체: `tda/data/<name>.py` 로더 + `configs/<name>.json` 추가 후
  `run(load_json('configs/<name>.json'), '<name>', ...)`.
- 충실도·가정·한계는 본 저장소 `README.md` / `docs/design.ko.md` 참고.